# Strong baseline: spoiler type classifier + extractive QA

This is the Milestone 2 strong baseline. It follows the setup of Hagen et al. (2022) and the SemEval-2023 Task 5 organizer baseline, written as our own code on top of Hugging Face `transformers`:

1. **Type classifier.** RoBERTa-base reads the post and the start of the article and predicts *phrase*, *passage* or *multi*.
2. **Extractive QA.** `deepset/roberta-base-squad2` is fine-tuned to find the spoiler in the article, with the post as the question. We train it with **three learning rates (1e-5, 3e-5, 5e-5)**; that sweep is the parameter study the final report asks for.
3. **Prediction.** The best learning rate (by dev BLEU) is used. Two decoding settings are tuned on dev: the longest answer allowed per type, and how many spans to return for multi spoilers. Then we predict dev and test, using the classifier's types.
4. **Scoring** with `score.py` (official SemEval-2023 metrics).

**Running it on the L4 VM** (about 1 hour):
```bash
cd milestone2
pip install -r requirements.txt
sudo apt-get install -y default-jre && bash setup_meteor.sh   # official METEOR; optional
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 strong-baseline.ipynb
```
Or open it in JupyterLab and run all cells. Run the `nbconvert` line inside `tmux` so it survives a dropped SSH connection.

**Smoke test** (a few seconds per step, on a laptop CPU): `SMOKE=1 jupyter nbconvert --to notebook --execute strong-baseline.ipynb --output smoke-run.ipynb`. It trains for 2 steps on 32 posts and writes everything under `smoke/`.

**Outputs:** checkpoints in `runs/` (not committed); predictions, the sweep table and scores in `output/`.

## Setup

Imports and run settings. The shared helpers live in the `spoilerqa/` package next to this notebook:
- `data.py`: loading the splits and locating spoiler spans in the article;
- `qa.py`: sliding windows and turning model scores into spoilers;
- `metrics.py`: the official metrics.

`SMOKE=1` in the environment switches to the tiny test run.

In [1]:
import gc
import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score
from transformers import (AutoModelForQuestionAnswering, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)
from transformers.utils import logging as hf_logging

from spoilerqa.data import DEFAULT_DATA_DIR, TYPES, load_split, post_text, spoiler_type, write_jsonl
from spoilerqa.metrics import as_text, bleu, ensure_nltk_data
from spoilerqa.qa import (DEFAULT_MAX_ANSWER_TOKENS, FeatureDataset, candidate_spans, decode, inference_features,
                          training_examples, training_features)

hf_logging.set_verbosity_error()
ensure_nltk_data()
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE", "0") == "1"
DATA_DIR = Path(os.environ.get("DATA_DIR", DEFAULT_DATA_DIR))
WORK_DIR = Path("smoke") if SMOKE else Path(".")
RUNS_DIR, OUTPUT_DIR = WORK_DIR / "runs", WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

CLASSIFIER_MODEL = "roberta-base"
QA_MODEL = "deepset/roberta-base-squad2"
QA_LEARNING_RATES = [1e-5, 3e-5, 5e-5]
MAX_LENGTH, STRIDE = 384, 128

# Smoke-test limits (None / -1 = use everything).
MAX_TRAIN = 32 if SMOKE else None
MAX_EVAL = 16 if SMOKE else None
MAX_STEPS = 2 if SMOKE else -1
USE_CPU = SMOKE
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported() and not USE_CPU

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| bf16:", USE_BF16, "| smoke test:", SMOKE, "| data:", DATA_DIR)

/home/dhanush/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/dhanush/project/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/dhanush/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


GPU: NVIDIA L4 | bf16: True | smoke test: False | data: /home/dhanush/project/data/splits


Load the splits. In the smoke test we keep only the first few posts of each.

In [2]:
train = load_split(DATA_DIR, "train")[:MAX_TRAIN]
dev = load_split(DATA_DIR, "dev")[:MAX_EVAL]
test = load_split(DATA_DIR, "test")[:MAX_EVAL]
print({"train": len(train), "dev": len(dev), "test": len(test)})

{'train': 3157, 'dev': 794, 'test': 1000}


Two small helpers used by both models.
- `make_training_args` holds the settings shared by every run: AdamW with weight decay 0.01, a linear schedule with 10% warmup, bf16 on the GPU, evaluation and a checkpoint after every epoch, and keeping the best epoch.
- `free_gpu` releases GPU memory between runs.

In [3]:
def make_training_args(output_dir, learning_rate, epochs, batch_size, metric, grad_accum=1, eval_batch_size=64):
    per_epoch = MAX_STEPS < 0
    return TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=learning_rate,
        num_train_epochs=epochs,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        per_device_eval_batch_size=eval_batch_size,
        weight_decay=0.01,
        warmup_steps=0.1,  # a float below 1 means a ratio of the total steps
        lr_scheduler_type="linear",
        eval_strategy="epoch" if per_epoch else "steps",
        eval_steps=None if per_epoch else MAX_STEPS,
        save_strategy="epoch" if per_epoch else "steps",
        save_steps=None if per_epoch else MAX_STEPS,
        save_total_limit=2,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model=metric,
        greater_is_better=True,
        logging_steps=50,
        bf16=USE_BF16,
        use_cpu=USE_CPU,
        report_to="none",
        seed=SEED,
    )


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Part 1: spoiler type classifier

The input is the post paired with the title and article, cut at 512 tokens. Multi spoilers are rare (17% of train) and the task is scored with balanced accuracy, so the loss weights each class by inverse frequency. The best of 5 epochs is chosen by dev balanced accuracy.

In [4]:
def encode_for_classifier(tokenizer, records, max_length=512):
    articles = ["\n".join([r["targetTitle"], *r["targetParagraphs"]]) for r in records]
    enc = tokenizer([post_text(r) for r in records], articles, truncation="only_second", max_length=max_length)
    enc["labels"] = [TYPES.index(spoiler_type(r)) for r in records]
    return enc


class WeightedTrainer(Trainer):
    # Trainer with an inverse-frequency class-weighted cross-entropy loss.
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device, outputs.logits.dtype)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss


def classifier_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return {"balanced_accuracy": balanced_accuracy_score(eval_pred.label_ids, preds),
            "macro_f1": f1_score(eval_pred.label_ids, preds, average="macro")}

Train the classifier (learning rate 2e-5, batch 16, 5 epochs) and save the best epoch to `runs/type-classifier/best`.

In [5]:
set_seed(SEED)
clf_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL)
clf_model = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_MODEL, num_labels=len(TYPES), id2label=dict(enumerate(TYPES)), label2id={t: i for i, t in enumerate(TYPES)})

counts = Counter(spoiler_type(r) for r in train)
class_weights = torch.tensor([len(train) / (len(TYPES) * max(counts[t], 1)) for t in TYPES])
print("class weights:", {t: round(w, 3) for t, w in zip(TYPES, class_weights.tolist())})

clf_keys = ["input_ids", "attention_mask", "labels"]
clf_trainer = WeightedTrainer(
    model=clf_model,
    args=make_training_args(RUNS_DIR / "type-classifier", 2e-5, epochs=5, batch_size=16, metric="balanced_accuracy"),
    train_dataset=FeatureDataset(encode_for_classifier(clf_tokenizer, train), clf_keys),
    eval_dataset=FeatureDataset(encode_for_classifier(clf_tokenizer, dev), clf_keys),
    processing_class=clf_tokenizer,
    data_collator=DataCollatorWithPadding(clf_tokenizer),
    compute_metrics=classifier_metrics,
    class_weights=class_weights,
)
clf_trainer.train()
clf_trainer.save_model(str(RUNS_DIR / "type-classifier" / "best"))

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4294.25it/s]


class weights: {'phrase': 0.781, 'passage': 0.836, 'multi': 1.913}
{'loss': '1.101', 'grad_norm': '2.314', 'learning_rate': '9.899e-06', 'epoch': '0.2525'}
{'loss': '1.098', 'grad_norm': '5.919', 'learning_rate': '2e-05', 'epoch': '0.5051'}
{'loss': '1.042', 'grad_norm': '7.417', 'learning_rate': '1.888e-05', 'epoch': '0.7576'}
{'eval_loss': '0.9082', 'eval_balanced_accuracy': '0.5145', 'eval_macro_f1': '0.4402', 'eval_runtime': '3.292', 'eval_samples_per_second': '241.2', 'eval_steps_per_second': '3.949', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]


{'loss': '0.9603', 'grad_norm': '12.26', 'learning_rate': '1.776e-05', 'epoch': '1.01'}
{'loss': '0.9492', 'grad_norm': '14.55', 'learning_rate': '1.663e-05', 'epoch': '1.263'}
{'loss': '0.9514', 'grad_norm': '6.017', 'learning_rate': '1.551e-05', 'epoch': '1.515'}
{'loss': '0.8508', 'grad_norm': '5.936', 'learning_rate': '1.439e-05', 'epoch': '1.768'}
{'eval_loss': '0.7908', 'eval_balanced_accuracy': '0.6555', 'eval_macro_f1': '0.6697', 'eval_runtime': '3.379', 'eval_samples_per_second': '235', 'eval_steps_per_second': '3.847', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.65it/s]


{'loss': '0.8902', 'grad_norm': '12.21', 'learning_rate': '1.327e-05', 'epoch': '2.02'}
{'loss': '0.7422', 'grad_norm': '19.34', 'learning_rate': '1.214e-05', 'epoch': '2.273'}
{'loss': '0.6945', 'grad_norm': '25.46', 'learning_rate': '1.102e-05', 'epoch': '2.525'}
{'loss': '0.6598', 'grad_norm': '10.13', 'learning_rate': '9.899e-06', 'epoch': '2.778'}
{'eval_loss': '0.6703', 'eval_balanced_accuracy': '0.7365', 'eval_macro_f1': '0.7154', 'eval_runtime': '3.398', 'eval_samples_per_second': '233.6', 'eval_steps_per_second': '3.825', 'epoch': '3'}


... (3 outputs left out)

{'loss': '0.4452', 'grad_norm': '5.875', 'learning_rate': '4.287e-06', 'epoch': '4.04'}
{'loss': '0.3773', 'grad_norm': '16.55', 'learning_rate': '3.165e-06', 'epoch': '4.293'}
{'loss': '0.3358', 'grad_norm': '23.71', 'learning_rate': '2.043e-06', 'epoch': '4.545'}
{'loss': '0.3738', 'grad_norm': '46.31', 'learning_rate': '9.203e-07', 'epoch': '4.798'}
{'eval_loss': '0.7853', 'eval_balanced_accuracy': '0.7321', 'eval_macro_f1': '0.7266', 'eval_runtime': '3.373', 'eval_samples_per_second': '235.4', 'eval_steps_per_second': '3.855', 'epoch': '5'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]


{'train_runtime': '239.8', 'train_samples_per_second': '65.84', 'train_steps_per_second': '4.129', 'train_loss': '0.6971', 'epoch': '5'}


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.34s/it]


Predict the type for every dev and test post. The predictions go to `output/type-classifier-{dev,test}.jsonl`, and part 3 uses them to decide how each spoiler is decoded.

In [6]:
predicted_types = {}
for split, records in [("dev", dev), ("test", test)]:
    output = clf_trainer.predict(FeatureDataset(encode_for_classifier(clf_tokenizer, records), clf_keys))
    probs = torch.softmax(torch.tensor(output.predictions).float(), dim=-1).numpy()
    predicted_types[split] = [TYPES[i] for i in probs.argmax(axis=-1)]
    gold = [spoiler_type(r) for r in records]
    print(f"{split}: balanced accuracy {balanced_accuracy_score(gold, predicted_types[split]):.4f}")
    write_jsonl([{"uuid": r["uuid"], "spoilerType": t, "probabilities": dict(zip(TYPES, map(float, p)))}
                 for r, t, p in zip(records, predicted_types[split], probs)],
                OUTPUT_DIR / f"type-classifier-{split}.jsonl")

del clf_trainer, clf_model
free_gpu()

dev: balanced accuracy 0.7365
test: balanced accuracy 0.6920


## Part 2: extractive QA, three learning rates

The context is the title followed by the paragraphs, one per line. Articles are longer than one model input, so the post + context is split into windows of 384 tokens that overlap by 128.
- Each training window is labeled with the answer's start and end tokens, or with the first token if the window doesn't contain the answer (the usual convention for sliding-window QA).
- Multi spoilers become one training example per span.
- The few spans we can't locate in the article are skipped.

In [7]:
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL)
examples, skipped = training_examples(train)
train_enc = training_features(qa_tokenizer, examples, MAX_LENGTH, STRIDE)
qa_keys = [k for k in ["input_ids", "attention_mask", "token_type_ids"] if k in train_enc]
qa_train_ds = FeatureDataset(train_enc, qa_keys + ["start_positions", "end_positions"])
dev_enc, dev_contexts = inference_features(qa_tokenizer, dev, MAX_LENGTH, STRIDE)
qa_dev_ds = FeatureDataset(dev_enc, qa_keys)

answered = sum(s > 0 for s in train_enc["start_positions"])
print(f"train: {len(train)} posts -> {len(examples)} spans ({skipped} not found in the article) -> "
      f"{len(qa_train_ds)} windows ({answered} contain the answer)")
print(f"dev: {len(dev)} posts -> {len(qa_dev_ds)} windows")

train: 3157 posts -> 4533 spans (14 not found in the article) -> 15940 windows (5385 contain the answer)
dev: 794 posts -> 2408 windows


To pick the best epoch we need a dev score after every epoch, and a loss doesn't tell us much about spoiler quality. So this trainer's evaluation decodes the dev spoilers and reports **BLEU**. It decodes with the *gold* types, so this score measures extraction only; the classifier comes in at prediction time.

In [8]:
class SpoilerQATrainer(Trainer):
    # Trainer whose evaluation decodes dev spoilers and reports BLEU instead of a loss.
    def __init__(self, *args, dev_records=None, dev_enc=None, dev_contexts=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.dev_records, self.dev_enc, self.dev_contexts = dev_records, dev_enc, dev_contexts

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        loader = self.get_eval_dataloader(eval_dataset or self.eval_dataset)
        output = self.evaluation_loop(loader, description="Decoding dev", prediction_loss_only=False,
                                      metric_key_prefix=metric_key_prefix)
        start_logits, end_logits = output.predictions
        candidates = candidate_spans(self.dev_enc, self.dev_contexts, start_logits, end_logits)
        preds = [as_text(decode(c, ctx, spoiler_type(r)))
                 for c, ctx, r in zip(candidates, self.dev_contexts, self.dev_records)]
        truths = [as_text(r["spoiler"]) for r in self.dev_records]
        metrics = {f"{metric_key_prefix}_bleu": bleu(truths, preds)}
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics

One training run for one learning rate: batch 16, 3 epochs, keeping the epoch with the best dev BLEU. Each run saves its best model to `runs/qa-roberta-base-lr<lr>/best` and its scores to `run_info.json` in the same folder.

In [9]:
def train_qa(learning_rate, epochs=3, batch_size=16):
    set_seed(SEED)
    run_dir = RUNS_DIR / f"qa-roberta-base-lr{learning_rate:g}"
    model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL)
    trainer = SpoilerQATrainer(
        model=model,
        args=make_training_args(run_dir, learning_rate, epochs, batch_size, metric="bleu"),
        train_dataset=qa_train_ds, eval_dataset=qa_dev_ds, processing_class=qa_tokenizer,
        dev_records=dev, dev_enc=dev_enc, dev_contexts=dev_contexts,
    )
    trainer.train()
    trainer.save_model(str(run_dir / "best"))
    per_epoch = [h["eval_bleu"] for h in trainer.state.log_history if "eval_bleu" in h]
    info = {"learning_rate": learning_rate, "dev_bleu_gold_types": max(per_epoch),
            "dev_bleu_per_epoch": per_epoch, "run_dir": str(run_dir)}
    (run_dir / "run_info.json").write_text(json.dumps(info, indent=2) + "\n")
    del trainer, model
    free_gpu()
    return info

**The learning-rate sweep.** Each run takes about 10 minutes on the L4. The table shows dev BLEU after every epoch and is saved to `output/qa-sweep.json` for the report.

In [10]:
sweep = []
for lr in QA_LEARNING_RATES:
    print(f"=== learning rate {lr:g} ===")
    sweep.append(train_qa(lr))

(OUTPUT_DIR / "qa-sweep.json").write_text(json.dumps(sweep, indent=2) + "\n")
pd.DataFrame(sweep)[["learning_rate", "dev_bleu_gold_types", "dev_bleu_per_epoch"]]

=== learning rate 1e-05 ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3636.32it/s]


{'loss': '2.157', 'grad_norm': '28.1', 'learning_rate': '1.633e-06', 'epoch': '0.05015'}
{'loss': '1.754', 'grad_norm': '23.94', 'learning_rate': '3.3e-06', 'epoch': '0.1003'}
{'loss': '1.554', 'grad_norm': '26.16', 'learning_rate': '4.967e-06', 'epoch': '0.1505'}
{'loss': '1.379', 'grad_norm': '28.59', 'learning_rate': '6.633e-06', 'epoch': '0.2006'}
{'loss': '1.509', 'grad_norm': '14.43', 'learning_rate': '8.3e-06', 'epoch': '0.2508'}
{'loss': '1.503', 'grad_norm': '24.21', 'learning_rate': '9.967e-06', 'epoch': '0.3009'}
{'loss': '1.401', 'grad_norm': '28.88', 'learning_rate': '9.818e-06', 'epoch': '0.3511'}
{'loss': '1.415', 'grad_norm': '53.19', 'learning_rate': '9.632e-06', 'epoch': '0.4012'}
{'loss': '1.363', 'grad_norm': '18.49', 'learning_rate': '9.446e-06', 'epoch': '0.4514'}
{'loss': '1.33', 'grad_norm': '19.37', 'learning_rate': '9.26e-06', 'epoch': '0.5015'}
{'loss': '1.304', 'grad_norm': '27.51', 'learning_rate': '9.075e-06', 'epoch': '0.5517'}
{'loss': '1.342', 'grad_nor

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]


{'loss': '1.404', 'grad_norm': '24.34', 'learning_rate': '7.402e-06', 'epoch': '1.003'}
{'loss': '1.077', 'grad_norm': '22.83', 'learning_rate': '7.217e-06', 'epoch': '1.053'}
{'loss': '1.078', 'grad_norm': '18.43', 'learning_rate': '7.031e-06', 'epoch': '1.103'}
{'loss': '1.148', 'grad_norm': '25.07', 'learning_rate': '6.845e-06', 'epoch': '1.153'}
{'loss': '1.062', 'grad_norm': '14.67', 'learning_rate': '6.659e-06', 'epoch': '1.204'}
{'loss': '1.123', 'grad_norm': '25.99', 'learning_rate': '6.473e-06', 'epoch': '1.254'}
{'loss': '1.098', 'grad_norm': '16.77', 'learning_rate': '6.288e-06', 'epoch': '1.304'}
{'loss': '0.9914', 'grad_norm': '20.45', 'learning_rate': '6.102e-06', 'epoch': '1.354'}
{'loss': '1.014', 'grad_norm': '23.68', 'learning_rate': '5.916e-06', 'epoch': '1.404'}
{'loss': '1.119', 'grad_norm': '22.28', 'learning_rate': '5.73e-06', 'epoch': '1.454'}
{'loss': '1.049', 'grad_norm': '26.2', 'learning_rate': '5.544e-06', 'epoch': '1.505'}
{'loss': '1.084', 'grad_norm': '2

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.73it/s]


... (21 outputs left out)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]


{'train_runtime': '518.6', 'train_samples_per_second': '92.21', 'train_steps_per_second': '5.767', 'train_loss': '1', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]


,learning_rate,dev_bleu_gold_types,dev_bleu_per_epoch
0,0.00001,0.443399,"[0.412279, 0.443399, 0.435084]"
1,0.00003,0.443220,"[0.418238, 0.428879, 0.44322]"
2,0.00005,0.427223,"[0.418559, 0.427223, 0.421387]"


## Part 3: prediction

Load the QA model with the best dev BLEU. Then score every dev and test window once, and keep all plausible answer spans per post (up to 128 tokens long), ranked by start + end score. Decoding later picks from these candidates, so trying different decoding settings needs no extra model calls.

In [11]:
best_run = max(sweep, key=lambda info: info["dev_bleu_gold_types"])
print("best learning rate:", best_run["learning_rate"], "| dev BLEU (gold types):", round(best_run["dev_bleu_gold_types"], 4))

device = "cpu" if USE_CPU else ("cuda" if torch.cuda.is_available() else "cpu")
qa_model = AutoModelForQuestionAnswering.from_pretrained(Path(best_run["run_dir"]) / "best").to(device).eval()


def score_windows(enc, batch_size=64):
    starts, ends = [], []
    with torch.no_grad():
        for i in range(0, len(enc["input_ids"]), batch_size):
            batch = {k: torch.tensor(enc[k][i:i + batch_size]).to(device) for k in qa_keys}
            out = qa_model(**batch)
            starts.append(out.start_logits.float().cpu().numpy())
            ends.append(out.end_logits.float().cpu().numpy())
    return np.concatenate(starts), np.concatenate(ends)


candidates, contexts = {}, {}
for split, records in [("dev", dev), ("test", test)]:
    enc, contexts[split] = inference_features(qa_tokenizer, records, MAX_LENGTH, STRIDE)
    candidates[split] = candidate_spans(enc, contexts[split], *score_windows(enc), max_answer_tokens=128)
print({split: len(c) for split, c in candidates.items()}, "posts with candidates")

best learning rate: 1e-05 | dev BLEU (gold types): 0.4434


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3548.39it/s]


{'dev': 794, 'test': 1000} posts with candidates


**Tuning decoding on dev.** Two settings matter:
- **Longest answer allowed per type.** The usual default of 30 tokens would cut off 29% of passage spoilers, while phrase spoilers are almost always under 6 tokens.
- **Number of spans for multi.** Annotations seem to stop at five spans.

For each predicted type we pick the setting with the best dev BLEU over the posts predicted as that type, using the classifier's types as at test time. Test is never used here.

In [12]:
CAP_GRID = {"phrase": [8, 16, 24, 32], "passage": [48, 64, 96, 128], "multi": [24, 48, 64]}
K_GRID = [1, 2, 3, 4, 5, 6]

settings = {"max_answer_tokens": dict(DEFAULT_MAX_ANSWER_TOKENS), "k": 5}
tuning = []
for t in TYPES:
    idx = [i for i, p in enumerate(predicted_types["dev"]) if p == t]
    if not idx:
        continue
    truths = [as_text(dev[i]["spoiler"]) for i in idx]
    for cap in CAP_GRID[t]:
        for k in (K_GRID if t == "multi" else [settings["k"]]):
            caps = {**settings["max_answer_tokens"], t: cap}
            preds = [as_text(decode(candidates["dev"][i], contexts["dev"][i], t, caps, k)) for i in idx]
            tuning.append({"type": t, "max_answer_tokens": cap, "k": k if t == "multi" else None,
                           "dev_posts": len(idx), "dev_bleu": bleu(truths, preds)})
    best = max((row for row in tuning if row["type"] == t), key=lambda row: row["dev_bleu"])
    settings["max_answer_tokens"][t] = best["max_answer_tokens"]
    if t == "multi":
        settings["k"] = best["k"]

settings.update({"qa_model": QA_MODEL, "learning_rate": best_run["learning_rate"]})
(OUTPUT_DIR / "strong-baseline-settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print(json.dumps(settings, indent=2))
pd.DataFrame(tuning).sort_values(["type", "dev_bleu"], ascending=[True, False]).groupby("type").head(3)

{
  "max_answer_tokens": {
    "phrase": 24,
    "passage": 96,
    "multi": 24
  },
  "k": 5,
  "qa_model": "deepset/roberta-base-squad2",
  "learning_rate": 1e-05
}


,type,max_answer_tokens,k,dev_posts,dev_bleu
12,multi,24,5.0,191,0.289009
18,multi,48,5.0,191,0.282135
24,multi,64,5.0,191,0.273418
6,passage,96,NaN,316,0.313008
4,passage,48,NaN,316,0.312980
5,passage,64,NaN,316,0.312011
2,phrase,24,NaN,287,0.581089
3,phrase,32,NaN,287,0.581080
1,phrase,16,NaN,287,0.574596


Write the predictions in the SemEval run format (`uuid`, `spoilerType`, `spoiler`):
- `strong-baseline-{dev,test}.jsonl` uses the classifier's types. **This is the strong baseline.**
- `strong-baseline-goldtypes-test.jsonl` decodes with the gold types instead. It's only for analysis: it shows how much the type classifier's mistakes cost.

In [13]:
def predictions(records, split, types):
    return [{"uuid": r["uuid"], "spoilerType": t,
             "spoiler": decode(c, ctx, t, settings["max_answer_tokens"], settings["k"])}
            for r, c, ctx, t in zip(records, candidates[split], contexts[split], types)]


write_jsonl(predictions(dev, "dev", predicted_types["dev"]), OUTPUT_DIR / "strong-baseline-dev.jsonl")
write_jsonl(predictions(test, "test", predicted_types["test"]), OUTPUT_DIR / "strong-baseline-test.jsonl")
write_jsonl(predictions(test, "test", [spoiler_type(r) for r in test]), OUTPUT_DIR / "strong-baseline-goldtypes-test.jsonl")
print(sorted(p.name for p in OUTPUT_DIR.glob("strong-baseline*")))

['strong-baseline-dev.jsonl', 'strong-baseline-goldtypes-test.jsonl', 'strong-baseline-settings.json', 'strong-baseline-test.jsonl']


## Part 4: scores

We run `score.py`, the Milestone 2 evaluation script, exactly as from the command line. It reports BLEU (the main number), METEOR, BERTScore, exact match and token F1, overall and per type, plus type balanced accuracy. The JSON files go to `output/scores/`. The smoke test skips BERTScore and scores only the posts it predicted.

In [14]:
def gold_file(split, records):
    if not SMOKE:
        return DATA_DIR / f"{split}.jsonl"
    path = WORK_DIR / f"gold-{split}.jsonl"  # the smoke test predicts only a few posts
    write_jsonl(records, path)
    return path


def run_score(split, records, name):
    cmd = [sys.executable, "score.py", "--gold", str(gold_file(split, records)),
           "--pred", str(OUTPUT_DIR / f"{name}-{split}.jsonl"),
           "--output", str(OUTPUT_DIR / "scores" / f"{name}-{split}.json")]
    if SMOKE:
        cmd.append("--no-bertscore")
    print(f"$ python score.py --gold {cmd[3]} --pred {cmd[5]}")
    print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)


run_score("dev", dev, "strong-baseline")
run_score("test", test, "strong-baseline")
run_score("test", test, "strong-baseline-goldtypes")

$ python score.py --gold /home/dhanush/project/data/splits/dev.jsonl --pred output/strong-baseline-dev.jsonl
Scored 794 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                    794       333       318       143
bleu              0.4041    0.5669    0.2683    0.3272
meteor            0.4366    0.2950    0.4066    0.5425
bertscore         0.9150    0.9403    0.8942    0.9024
exact-match       0.2884    0.5105    0.1352    0.1119
token-f1          0.5133    0.6367    0.3887    0.5028

spoiler type: balanced accuracy 0.7365
  phrase   precision 0.7840  recall 0.6757  f1 0.7258
  passage  precision 0.7342  recall 0.7296  f1 0.7319
  multi    precision 0.6021  recall 0.8042  f1 0.6886

BLEU (all): 0.4041

$ python score.py --gold /home/dhanush/project/data/splits/test.jsonl --pred output/strong-baseline-test.jsonl
Scored 1000 posts (0 missing predictions)

spoiler              all    phrase   passage     multi
n                   1000       4

## A few test predictions

Six random test posts with their gold and predicted spoilers, to sanity-check the numbers above.

In [15]:
test_preds = {p["uuid"]: p for p in predictions(test, "test", predicted_types["test"])}
sample = pd.DataFrame([{
    "post": post_text(r), "gold type": spoiler_type(r), "predicted type": test_preds[r["uuid"]]["spoilerType"],
    "gold spoiler": as_text(r["spoiler"]), "predicted spoiler": as_text(test_preds[r["uuid"]]["spoiler"]),
} for r in test])
sample.sample(min(6, len(sample)), random_state=0)

,post,gold type,predicted type,gold spoiler,predicted spoiler
993,Inside our three-month effort to attend Apple's iPhone 7 launch party,passage,passage,"The truth though is that large tech companies, especially in Silicon Valley, often use access to...",September 6
859,Mum takes 'funny' photo of toddler - breaks down over chilling truth behind it,passage,passage,"""She was practising for a lockdown drill at her preschool and what you should do if you are stuc...",practising hiding from a gunman
298,Duterte plans to ‘SHOOT and KILL’ these people; find out who they are,passage,multi,business owners that he will kill them if they continue to practice endo or contractualization,President Duterte warned business owners that he will kill them if they continue to practice end...
553,LOOK: This Is Definitely The Biggest Spring 2014 Trend,passage,phrase,popping Spring colors and mixed prints,mixed prints
672,Lot of Pokémon Go players demanding refunds after latest update due to this reason,passage,passage,"game would no longer be supported in unlocked, rooted or jail-broken devices","no longer be supported in unlocked, rooted or jail-broken devices was already given by Niantic b..."
971,"Former ""SNL"" star heads to ""House of Lies""",phrase,phrase,Fred Armisen,Fred Armisen
